# QMCPy Performance Optimizations Demo

Sou-Cheng T. Choi

Illinois Institute of Technology and SouLab LLC.

Modification date: 9/12/2026

Creation date: 8/14/2026

For reproducibility, this notebook was run with:
- Python 3.13.13, NumPy 2.5.0, SciPy 1.17.1, QMCPy 2.4, PyTorch (for the multitask kernel section)
- OS: macOS 15.6.1

This notebook benchmarks a set of small, targeted optimizations applied to QMCPy's internals: replacing generic `scipy.stats` distribution-object calls with the underlying `scipy.special` C-level functions, and replacing `np.einsum` calls (run with its default, non-BLAS-routed contraction path) with `@`/`matmul`.

**Why these are fast:**

- `scipy.stats.norm.ppf(x)` is a method on a generic `rv_continuous` distribution object. Even though it ultimately calls the same underlying Cephes routine, it first pays for loc/scale handling, support-bounds validation, and edge-case masking. `scipy.special.ndtri(x)` (and `ndtr(x)` for the forward CDF) call that routine directly.
- `np.einsum(subscripts, A, B)` is a general-purpose tensor-contraction *interpreter*. Unless called with `optimize=True`, it does not automatically recognize that a given contraction is really just a matrix multiply, so it doesn't route through BLAS's `GEMM` kernel the way `@`/`np.matmul` does. For contractions that *are* matrix multiplies in disguise, this can be a 10-50x difference.

Each section below defines a small "legacy" function using the old approach, times it against the equivalent QMCPy call (which now uses the fast approach), and reports the speedup. All outputs are verified numerically identical (up to floating-point rounding) before timing.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMCSoftware/blob/develop/demos/performance_optimizations_demo.ipynb)

In [1]:
# @title Execute this cell to install dependencies
try:
  import google.colab
  IN_COLAB = True
except ImportError:
  IN_COLAB = False
if IN_COLAB:
  !pip install -q qmcpy


In [2]:
import time
import numpy as np
from scipy.stats import norm
from scipy.stats import t as t_dist
from scipy.special import ndtri, ndtr, stdtr, stdtrit

import qmcpy as qp
from qmcpy import (
    BayesianLRCoeffs,
    BrownianMotion,
    DigitalNetB2,
    GaussianCopula,
    GeometricBrownianMotion,
    KernelGaussian,
    KernelMultiTask,
)

rng = np.random.default_rng(7)

def bench(f, target_time=0.05, max_reps=5000):
    f()  # warm up
    reps = 5
    while True:
        times = np.empty(reps)
        for i in range(reps):
            t0 = time.perf_counter()
            result = f()
            times[i] = time.perf_counter() - t0
        # Report min-over-reps once total measured time clears target_time: for
        # sub-millisecond calls, a few reps of a mean-based timer are dominated by
        # scheduler/GC jitter rather than true cost, which can flip which side "wins".
        if times.sum() >= target_time or reps >= max_reps:
            return result, times.min()
        reps *= 4

results_table = []

def report(name, t_old, t_new, max_diff):
    speedup = t_old / t_new
    results_table.append((name, t_old, t_new, speedup, max_diff))
    print(f'{name}')
    print(f'  old: {t_old:.5f} s   new: {t_new:.5f} s   speedup: {speedup:.1f}x   max diff: {max_diff:.2e}')


## 1. Gaussian PCA Transform (`qmcpy/true_measure/gaussian.py`)

This is the code path used by default (`decomp_type="PCA"`) by `Gaussian`, `BrownianMotion`, and `GeometricBrownianMotion` -- so it's exercised any time `demos/GBM/gbm_demo.ipynb` samples GBM paths with a Sobol', Lattice, or Halton sampler. `A` below stands in for the cached PCA factor matrix.

In [3]:
n, d = 2**14, 252
x = rng.random((n, d))
A = rng.standard_normal((d, d))
mu = np.zeros(d)

def legacy_gaussian_transform():
    return mu + np.einsum('...ij,kj->...ik', norm.ppf(x), A)

def fast_gaussian_transform():
    out = ndtri(x) @ A.T
    out += mu
    return out

r_old, t_old = bench(legacy_gaussian_transform)
r_new, t_new = bench(fast_gaussian_transform)
report('Gaussian PCA transform (gaussian.py)', t_old, t_new, np.abs(r_old - r_new).max())


Gaussian PCA transform (gaussian.py)
  old: 0.34742 s   new: 0.04209 s   speedup: 8.3x   max diff: 1.21e-13


## 2. Brownian Bridge Transform (`qmcpy/true_measure/brownian_motion.py`)

Used whenever `decomp_type="BrownianBridge"` -- the construction showcased in `demos/brownian_bridge.ipynb`. We compare `BrownianMotion`'s actual bridge transform against a copy of the same code with `ndtri` swapped back for `norm.ppf`.

Note this section's speedup is more modest than the others: `_bridge_transform` itself still runs an O(d) sequential Python loop over dimensions (unrelated to this session's changes), which dominates total runtime here and dilutes the `ndtri` win. The einsum/ndtri techniques shown in this notebook do not address that loop.

In [4]:
d, n_paths = 256, 2**12
bm = BrownianMotion(DigitalNetB2(d, seed=7), decomp_type='BrownianBridge')
u = rng.random((n_paths, d))

def legacy_bridge_transform():
    z = norm.ppf(u)
    w = bm._bridge_transform(z)
    paths = bm.drift_time_vec_plus_init + np.sqrt(bm.diffusion) * w
    return paths[..., bm._output_order]

def fast_bridge_transform():
    return bm._transform(u)

r_old, t_old = bench(legacy_bridge_transform)
r_new, t_new = bench(fast_bridge_transform)
report('Brownian bridge transform (brownian_motion.py)', t_old, t_new, np.abs(r_old - r_new).max())


Brownian bridge transform (brownian_motion.py)
  old: 0.06040 s   new: 0.03339 s   speedup: 1.8x   max diff: 0.00e+00


## 3. Gaussian Copula Density (`qmcpy/true_measure/gaussian_copula.py`)

Used by `GaussianCopula._weight()`, showcased in `demos/copula_examples.ipynb`. This combines *both* techniques: `norm.ppf` -> `ndtri`, and a 3-operand quadratic-form `einsum` -> a matmul-then-reduce.

In [5]:
d, n = 40, 2**14
M = rng.standard_normal((d, d)); M = M @ M.T  # stand-in for corr_inv_minus_eye
u = rng.random((n, d))
logdet = 0.0

def legacy_copula_quad():
    z = norm.ppf(u)
    quad = np.einsum('...i,ij,...j->...', z, M, z)
    return -0.5 * logdet - 0.5 * quad

def fast_copula_quad():
    z = ndtri(u)
    quad = ((z @ M) * z).sum(-1)
    return -0.5 * logdet - 0.5 * quad

r_old, t_old = bench(legacy_copula_quad)
r_new, t_new = bench(fast_copula_quad)
report('Gaussian copula log-density (gaussian_copula.py)', t_old, t_new, np.abs(r_old - r_new).max())


Gaussian copula log-density (gaussian_copula.py)
  old: 0.06248 s   new: 0.00634 s   speedup: 9.9x   max diff: 7.50e-12


## 4. Multitask Kernel Matrix (`qmcpy/kernel/multitask_kernel.py`)

`KernelMultiTask.taskmat` builds a batched task-covariance matrix for multi-output/multi-fidelity Bayesian cubature. It previously used `einsum("...ij,...kj->...ik", ...)`; now it uses a batched `matmul`.

In [6]:
batch, num_tasks, rank = 2**10, 8, 8
factor = rng.standard_normal((batch, num_tasks, rank))

def legacy_taskmat():
    return np.einsum('...ij,...kj->...ik', factor, factor)

def fast_taskmat():
    return np.matmul(factor, np.swapaxes(factor, -1, -2))

r_old, t_old = bench(legacy_taskmat)
r_new, t_new = bench(fast_taskmat)
report('Multitask kernel taskmat (multitask_kernel.py)', t_old, t_new, np.abs(r_old - r_new).max())


Multitask kernel taskmat (multitask_kernel.py)
  old: 0.00036 s   new: 0.00010 s   speedup: 3.6x   max diff: 7.11e-15


## 5. Bayesian Logistic Regression Integrand (`qmcpy/integrand/bayesian_lr_coeffs.py`)

`BayesianLRCoeffs.g(x)` evaluates the log-likelihood integrand at each sampled coefficient vector; showcased in `demos/vectorized_qmc_bayes.ipynb`. The `einsum("...j,ij->...i", x, feature_array)` step is really `x @ feature_array.T`.

In [7]:
n, n_coeffs, n_obs = 2**14, 30, 5
x = rng.standard_normal((n, n_coeffs))
feature_array = rng.standard_normal((n_obs, n_coeffs))

def legacy_bayesian_lr():
    return np.einsum('...j,ij->...i', x, feature_array)

def fast_bayesian_lr():
    return x @ feature_array.T

r_old, t_old = bench(legacy_bayesian_lr)
r_new, t_new = bench(fast_bayesian_lr)
report('Bayesian LR feature projection (bayesian_lr_coeffs.py)', t_old, t_new, np.abs(r_old - r_new).max())


Bayesian LR feature projection (bayesian_lr_coeffs.py)
  old: 0.00038 s   new: 0.00023 s   speedup: 1.6x   max diff: 1.07e-14


## 6. Student-t Quantile (`qmcpy/true_measure/student_t.py`, `student_t_copula.py`)

`StudentT` and `StudentTCopula` build multivariate Student-t samples via a sequence of univariate conditional quantiles, each previously computed with `scipy.stats.t.ppf`. `scipy.special.stdtrit` is the direct-call equivalent -- note the argument order flips from `t.ppf(q, df)` to `stdtrit(df, q)`.

Unlike `ndtri`, `stdtrit` is not a simple closed-form transform -- Cephes solves for it iteratively, so its own compute cost grows with array size and increasingly dominates the fixed `rv_continuous` overhead being removed. The win here is therefore much more size-dependent than in Sections 1-3: large for a handful of values, modest for a full batch of QMC points.

In [8]:
df = 6.0
n_paths = 2**10
p = rng.random(n_paths)

def legacy_t_quantile():
    return t_dist.ppf(p, df=df)

def fast_t_quantile():
    return stdtrit(df, p)

r_old, t_old = bench(legacy_t_quantile)
r_new, t_new = bench(fast_t_quantile)
report('Student-t quantile (student_t_copula.py)', t_old, t_new, np.abs(r_old - r_new).max())


Student-t quantile (student_t_copula.py)
  old: 0.00018 s   new: 0.00015 s   speedup: 1.2x   max diff: 0.00e+00


## 7. Black-Scholes Exact Price (`qmcpy/integrand/financial_option.py`)

`FinancialOption.get_exact_value()` prices European/Asian options with closed-form Black-Scholes-type formulas -- each a handful of *scalar* `norm.cdf` evaluations, called once per pricing request rather than once per QMC sample. This is effectively the opposite regime from Section 5's Bayesian LR case: the workload is tiny either way, but here `norm.cdf`'s `rv_continuous` overhead is the *entire* cost (there's no BLAS/vectorization to fall back on), so removing it wins by orders of magnitude rather than being noise-dominated.

In [9]:
d1, d2 = 0.35, 0.15  # stand-in for the two scalar log-moneyness terms in get_exact_value

def legacy_bs_cdf():
    return norm.cdf(d1) - norm.cdf(d2)

def fast_bs_cdf():
    return ndtr(d1) - ndtr(d2)

r_old, t_old = bench(legacy_bs_cdf)
r_new, t_new = bench(fast_bs_cdf)
report('Black-Scholes normal CDF (financial_option.py)', t_old, t_new, np.abs(r_old - r_new).max())


Black-Scholes normal CDF (financial_option.py)
  old: 0.00003 s   new: 0.00000 s   speedup: 258.4x   max diff: 0.00e+00


## Summary

In [10]:
import pandas as pd
summary_df = pd.DataFrame(
    results_table,
    columns=['Component', 'Old (s)', 'New (s)', 'Speedup', 'Max Abs Diff'],
)
sorted_summary_df = summary_df.sort_values(by='Speedup', ascending=False)
sorted_summary_df.round(4)


,Component,Old (s),New (s),Speedup,Max Abs Diff
6,Black-Scholes normal CDF (financial_option.py),0.0000,0.0000,258.4198,0.0
2,Gaussian copula log-density (gaussian_copula.py),0.0625,0.0063,9.8590,0.0
0,Gaussian PCA transform (gaussian.py),0.3474,0.0421,8.2533,0.0
3,Multitask kernel taskmat (multitask_kernel.py),0.0004,0.0001,3.5932,0.0
1,Brownian bridge transform (brownian_motion.py),0.0604,0.0334,1.8088,0.0
4,Bayesian LR feature projection (bayesian_lr_co...,0.0004,0.0002,1.6011,0.0
5,Student-t quantile (student_t_copula.py),0.0002,0.0001,1.2169,0.0


**Where these speedups show up in existing demos**, next time each is re-run:

- `demos/GBM/gbm_demo.ipynb`, `demos/GBM/gbm_examples.ipynb` -- GeometricBrownianMotion and BrownianMotion path generation (Section 1 and 2 above)
- `demos/brownian_bridge.ipynb` -- BrownianMotion with `decomp_type="BrownianBridge"` (Section 2)
- `demos/copula_examples.ipynb`, `demos/product_measure.ipynb` -- GaussianCopula (Section 3) and StudentTCopula (Section 6)
- `demos/vectorized_qmc.ipynb`, `demos/vectorized_qmc_bayes.ipynb` -- BayesianLRCoeffs (Section 5)
- `demos/pricing_options.ipynb`, `demos/asian-option-mlqmc.ipynb`, `demos/control_variates.ipynb` -- FinancialOption exact-value pricing (Section 7)

No existing demo currently exercises `KernelMultiTask` (Section 4); it is used internally by multi-output/multi-fidelity Bayesian cubature stopping criteria.

## Appendix: Other `scipy.stats` -> `scipy.special` swaps

The overhead-elimination trick above generalizes: any `scipy.stats.<dist>.{ppf,cdf,logcdf,...}` call has a `scipy.special` counterpart that skips the generic `rv_continuous`/`rv_discrete` dispatch (loc/scale handling, support-bounds validation, edge-case masking) and calls the underlying compiled routine directly.

| Instead of (`scipy.stats`, slow) | Use (`scipy.special`, fast) | Purpose | Used in QMCPy? |
|---|---|---|---|
| `norm.cdf` | `ndtr` | forward normal CDF | Yes -- `gaussian_copula.py`, `financial_option.py`, several stopping criteria (Sections 3, 7 above) |
| `norm.ppf` | `ndtri` | inverse normal CDF | Yes -- `gaussian.py`, `brownian_motion.py`, `johnsons_su.py`, several stopping criteria (Sections 1-2 above) |
| `norm.logcdf` | `log_ndtr` | log of normal CDF, numerically stable in the tails | Not currently used |
| `t.cdf` | `stdtr` | Student-t CDF | Yes -- `student_t_copula.py` |
| `t.ppf` | `stdtrit` | Student-t quantile (argument order flips to `stdtrit(df, p)`) | Yes -- `student_t.py`, `student_t_copula.py`, `cub_qmc_rep_student_t.py` (Section 6 above) |
| `chi2.ppf` / `.cdf` | `chdtriv` / `chdtr` | chi-squared quantile/CDF | Not currently used |
| `gamma.ppf` / `.cdf` | `gammainccinv` / `gammainc` (regularized incomplete gamma) | gamma quantile/CDF | Not currently used |
| `beta.ppf` / `.cdf` | `betaincinv` / `betainc` | beta quantile/CDF | Not applicable -- `kumaraswamy.py` looks similar but is not a Beta distribution; it has its own closed-form quantile, `(1-(1-x)**(1/b))**(1/a)`, already faster than any `scipy` Beta call |
| `poisson.cdf` | `pdtr` | Poisson CDF | Not currently used |
| `binom.cdf` | `bdtr` | binomial CDF | Not currently used |
| `f.cdf` / `.ppf` | `fdtr` / `fdtri` | F-distribution CDF/quantile | Not currently used |

Sections 1-3, 6, and 7 above benchmark the rows already used in QMCPy; the rest of the table is included for reference should a future distribution need one of these.

## References

$[1]$ Harris, C.R., Millman, K.J., van der Walt, S.J. et al. (2020). Array programming with NumPy. *Nature* 585, 357-362.

$[2]$ Virtanen, P., Gommers, R., Oliphant, T.E. et al. (2020). SciPy 1.0: Fundamental Algorithms for Scientific Computing in Python. *Nature Methods*, 17(3), 261-272.